# Iron man 2 — the v3.4 version (VEi) against a correctly built BC

**VEi** (the locked v3.4 version: small-canvas reference → SR → fal-canvas edit, no ankle cut) against **BC** (v3.1's incumbent, rebuilt properly: bald pass → head subtracted by the V2 cropper → edit), 200-pair matrix, seeds **46/47/48**. ~1,312 klein calls. **Self-contained**: the only image inputs are the raw person/garment photographs in the bundle's `testset/`; normalise, BiRefNet A4 crops, framing reads, and every klein call run fresh on this A100 (Drive supplies only model-weight caches).

**Two sessions (the second needs session 1's zip on Drive):**
- **Session 1 — Run all.** VEi refs + 600 VEi edits + 56 BC bald frames → zip to Drive.
- **Session 2 — run cells 1–4, then 7–8** (skip 5–6). Cell 7 builds the head-subtracted BC refs **on this A100** with the verbatim V2 cropper modules (validated against locally-made refs of record shipped in the bundle), then runs the 600 BC edits at the fal call-2 canvas.

Open directly: https://colab.research.google.com/github/101011101/magichour_takehome/blob/v3.3-lock/v3/colab/v34_a100.ipynb — Runtime → A100.


In [ ]:
# 1 · settings
A100_USD_PER_HOUR = 0.689     # CAD/h at 5.3 CU/h x CAD 0.13/CU; edit if your rate differs
SEEDS = [46, 47, 48]          # the iron-man seeds: comparable with the v3.3 record
ARMS = ("VEi",)               # the locked v3.4 version; BC handled per stage below
MATRICES = ["matrix.csv"]     # the 200-pair iron-man matrix
DRIVE_PROJECT_DIR = "Side projects and shi"

In [ ]:
# 2 · Drive: find the HF cache that holds klein; reuse the iron-man run
import os
from google.colab import drive
drive.mount('/content/drive')
MYDRIVE = '/content/drive/MyDrive'; BASE = os.path.join(MYDRIVE, DRIVE_PROJECT_DIR)
KLEIN = 'models--black-forest-labs--FLUX.2-klein-4B'
candidates = [os.path.join(MYDRIVE, 'hf_cache'), os.path.join(BASE, 'tryon_models', 'hf_cache'), os.path.join(BASE, 'hf_cache')]
found = [c for c in candidates if os.path.isdir(os.path.join(c, 'hub', KLEIN))]
os.environ['HF_HOME'] = found[0] if found else candidates[0]
os.environ['V3_MODEL_DIR'] = os.path.join(BASE if os.path.isdir(BASE) else MYDRIVE, 'v3_models')
os.makedirs(os.environ['HF_HOME'], exist_ok=True); os.makedirs(os.environ['V3_MODEL_DIR'], exist_ok=True)
print('HF_HOME =', os.environ['HF_HOME'], '(klein cached)' if found else '(no cached klein - cell 4 downloads ~13 GB once)')


In [ ]:
# 3 · install; pull the bundle from GitHub (public, branch v3.3-lock)
!pip -q install -U diffusers transformers accelerate sentencepiece protobuf mediapipe onnxruntime-gpu opencv-python-headless
!cd /content && rm -rf v34 && wget -q -O v33_ironman_bundle.zip https://github.com/101011101/magichour_takehome/raw/v3.3-lock/v33_ironman_bundle.zip && unzip -qo v33_ironman_bundle.zip -d v34
%cd /content/v34
import os, zipfile, onnxruntime as ort, torch
assert os.path.exists('realesr-general-x4v3.pth') and os.path.exists('lib/run_ironman.py') and os.path.exists('v34_failures.csv') and os.path.exists('v34_controls.csv'), 'bundle incomplete'
os.makedirs('run/inputs', exist_ok=True)   # self-contained: no previous run reused - normalise and A4 crops computed fresh from testset/ (raw photos in the bundle)
print('onnxruntime providers:', ort.get_available_providers(), '| gpu:', torch.cuda.get_device_name(0))

In [ ]:
# 4 · load klein once, timed
import sys; sys.path.insert(0, 'lib')
import klein_local as K
K.load(); K.info()

In [ ]:
# 5 · one pair first (session 1)
import run_ironman as R
R.main(MATRICES[0], 'testset', limit=1, seeds=SEEDS[:1], arms=ARMS, gpu_usd_per_hour=A100_USD_PER_HOUR)
print(sorted(f for f in os.listdir('run/gen')))


In [ ]:
# 6 · session 1: the full VEi arm, then BC's bald frames (resumable)
R.main(MATRICES[0], 'testset', limit=None, seeds=SEEDS, arms=ARMS, gpu_usd_per_hour=A100_USD_PER_HOUR)
R.main(MATRICES[0], 'testset', limit=None, seeds=SEEDS, arms=("BC",), gpu_usd_per_hour=A100_USD_PER_HOUR, stage="bald")
import json; print(json.dumps(json.load(open('run/meta/cost.json')), indent=1))


In [ ]:
# 7 · session 2: BC references on this A100 (V2 cropper, verbatim modules), then the BC edits
import glob, subprocess, zipfile as zf
import cv2, numpy as np
import run_ironman as R
# bald frames from the session-1 zip on Drive
s1 = sorted(glob.glob(os.path.join(BASE, 'v3_runs', 'v34_ironman2_*.zip')))
assert s1, 'no session-1 zip (v34_ironman2_*.zip) on Drive'
with zf.ZipFile(s1[-1]) as z:
    balds = [n for n in z.namelist() if n.endswith('__bald.jpg')]
    z.extractall('run', members=balds)
print(len(balds), 'bald frames from', os.path.basename(s1[-1]))
for f in glob.glob('run/refs/*__BC.jpg'): os.remove(f)   # never trust pre-existing BC refs
bcz = sorted(glob.glob(os.path.join(BASE, 'v3_runs', 'v34_bc_refs*.zip')))
if bcz:   # the local route, if its zip exists on Drive - takes precedence as the environment of record
    zf.ZipFile(bcz[-1]).extractall('run'); print('BC refs from the LOCAL cropper:', os.path.basename(bcz[-1]))
else:     # crop here: the same script, same modules, byte-for-byte, models self-download (~1.3 GB once)
    subprocess.run([sys.executable, 'lib/ironman_bc_crop.py', 'run'], check=True)
    # validate against the locally-made refs of record shipped in the bundle
    bad = []
    for vp in sorted(glob.glob('validation/*__BC.jpg')):
        stem = os.path.basename(vp)
        a, b = cv2.imread(vp), cv2.imread('run/refs/' + stem)
        if b is None: bad.append((stem, 'missing')); continue
        if abs(a.shape[0]-b.shape[0]) > 8 or abs(a.shape[1]-b.shape[1]) > 8: bad.append((stem, f'shape {a.shape[:2]} vs {b.shape[:2]}')); continue
        bb = cv2.resize(b, (a.shape[1], a.shape[0])); mad = float(np.abs(a.astype(np.float32)-bb.astype(np.float32)).mean())
        print(f'  validate {stem}: MAD {mad:.2f}')
        if mad > 4.0: bad.append((stem, f'MAD {mad:.2f}'))
    assert not bad, f'cropper validation failed vs the local refs of record: {bad}'
    print('cropper validated against', len(glob.glob('validation/*__BC.jpg')), 'local refs of record')
n = len(glob.glob('run/refs/*__BC.jpg')); assert n >= 56, f'only {n} BC refs'
R.main(MATRICES[0], 'testset', limit=None, seeds=SEEDS, arms=("BC",), gpu_usd_per_hour=A100_USD_PER_HOUR, stage="bcedit", bc_canvas="fal")


In [ ]:
# 7 · zip this run's references, outputs and meta to Drive
import shutil, time
name = f"v34_ironman2_{time.strftime('%Y%m%d_%H%M')}"
with zipfile.ZipFile(f'/content/{name}.zip', 'w', zipfile.ZIP_DEFLATED) as z:
    for f in os.listdir('run/refs'):
        if any(a in f for a in ('__VEi', '__bald', '__BC', '__VS', '__VA', '__VE', '__V34', '__Vnc', '__Vfc')): z.write('run/refs/' + f, 'refs/' + f)
    for f in os.listdir('run/gen'): z.write('run/gen/' + f, 'gen/' + f)
    for f in os.listdir('run/inputs'): z.write('run/inputs/' + f, 'inputs/' + f)
    for f in os.listdir('run/meta'): z.write('run/meta/' + f, 'meta/' + f)
os.makedirs(os.path.join(BASE, 'v3_runs'), exist_ok=True); shutil.copy(f'/content/{name}.zip', os.path.join(BASE, 'v3_runs', name + '.zip'))
print('->', os.path.join(BASE, 'v3_runs', name + '.zip'), ' then locally: python3 v3/build/v34_a100_page.py <unpacked dir> --arm V34')